In [1]:
!export CUTENSORNET_DEVICE=0
 

In [2]:
# Cell 1: must run before importing cudaq

import os
import subprocess

# Pin this notebook kernel to one GPU for tensornet-mps

# Avoid CPU thread oversubscription.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))

subprocess.run(["nvidia-smi", "-L"])

import cuquantum
print("tensornet in cuquantum:", hasattr(cuquantum, "tensornet"))

import cuquantum.tensornet as cutn
print("cutensornet loaded OK")

CUDA_VISIBLE_DEVICES = 0
GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-a5c72418-0740-c67c-f26f-0a3fde94c3d9)


/u/jjuradod/miniconda3/envs/cuquantum_env/lib/python3.11/site-packages/cupy/_environment.py:670: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy, cupy-cuda13x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


tensornet in cuquantum: True
cutensornet loaded OK


In [3]:
# from juliacall import Main as jl

# jl.seval("import Pkg")
# jl.seval('Pkg.add("PauliPropagation")')
# jl.seval("using PauliPropagation")
from qaoa_training_pipeline.evaluation import MPSEvaluator, EfficientDepthOneEvaluator
from qaoa_training_pipeline.training import DepthOneScanTrainer
from qaoa_training_pipeline.utils.graph_utils import load_graph, graph_to_operator
from qaoa_training_pipeline.evaluation.cuquantum_mps import CuQuantumMPSEvaluator
from qopt_best_practices.sat_mapping import SATMapper
from qiskit.transpiler.passes.routing.commuting_2q_gate_routing import SwapStrategy

import numpy as np
import time

import cupy as cp

cp.cuda.set_allocator(None)
cp.cuda.set_pinned_memory_allocator(None)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [ ]:
graph_path = "instances/line_to_full/000_10nodes_0swap_layers.json"
graph = load_graph(graph_path)
cost_op = graph_to_operator(graph, pre_factor=-0.5)

FileNotFoundError: [Errno 2] No such file or directory: '../instances/line_to_full/000_10nodes_0swap_layers.json'

In [ ]:
swap_strategy = SwapStrategy.from_line(range(graph.order()))

start = time.time()
sat_graph, _, layers = SATMapper(timeout=10).remap_graph_with_sat(graph=graph, swap_strategy=swap_strategy)
print(f"Mapped graph in {time.time() - start} sec")

Mapped graph in 0.03145575523376465 sec


In [ ]:
print(layers)
max(abs(u-v) for u,v in sat_graph.edges())

0


1

In [ ]:
trainer = DepthOneScanTrainer(EfficientDepthOneEvaluator())
results = trainer.train(cost_op, parameter_ranges=((0, np.pi/2), (0, np.pi/2)))
params =results["optimized_params"]
results["energy"]

2.2945448932995527

In [ ]:
sat_cost_op = graph_to_operator(sat_graph, pre_factor=-0.5)

In [ ]:
CPU_evaluator = MPSEvaluator(
        bond_dim_circuit=512,
        threshold_circuit=1e-10, 
        use_vidal_form=True, 
        use_swap_strategy=True,
        store_schmidt_values=True,
        store_intermediate_schmidt_values=True
)

In [ ]:
# Warm-up
time_start = time.time()
energy = CPU_evaluator.evaluate(sat_cost_op, params)
print(f"Energy computed in {time.time() - time_start}")
print(energy)
# Timed loop
times = []
for _ in range(2):
    t0 = time.time()
    _ = CPU_evaluator.evaluate(sat_cost_op, params)
    times.append(time.time() - t0)

print("mean steady-state:", np.mean(times))
print("median steady-state:", np.median(times))

/u/jjuradod/miniconda3/envs/cuquantum_env/lib/python3.11/site-packages/cotengra/hyperoptimizers/hyper.py:36: UserWarning: Couldn't import `kahypar` - skipping from default hyper optimizer and using basic `labels` method instead. `kahypar` is highly recommended for the best quality contraction paths.
  warnings.warn(


Energy computed in 2.860368013381958
2.2945448932995505
mean steady-state: 0.01857280731201172
median steady-state: 0.01857280731201172


In [ ]:
cuQuantum_evaluator = CuQuantumMPSEvaluator(max_bond_dim=2, rel_cutoff=0, abs_cutoff=None, precision="fp32", mpo_application="approximate", gauge_option="free", normalization=None, svd_algo="gesvd", use_swap_strategy=True, mode="tn")
# Warm-up
time_start = time.time()
energy = np.abs(cuQuantum_evaluator.evaluate(sat_cost_op, np.array(params)))
print(f"Energy computed in {time.time() - time_start}")
print(energy)
# Timed loop
times = []
for _ in range(2):
    t0 = time.time()
    _ = cuQuantum_evaluator.evaluate(sat_cost_op, params)
    times.append(time.time() - t0)

print("mean steady-state:", np.mean(times))
print("median steady-state:", np.median(times))
cuQuantum_evaluator.free_state()

CUDAError: CUDA_ERROR_UNKNOWN: This indicates that an unknown internal error has occurred.